### Libraries

In [ ]:
# !pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126

In [ ]:
import torch
import numpy as np

### Numpy

Python library for numerical computing. Faster than plain Python lists for numerical tasks

In [ ]:
data = [[1, 2],[3, 4]]
np_array = np.array(data)

print(np_array)

### Tensor

A Tensor is a multi-dimensional matrix containing elements of a single data type. Everything in PyTorch is based on Tensor operations.

In [ ]:
x_data = torch.tensor(data)

print(x_data)

Attributes of a Tensor

Tensor attributes describe their shape, datatype, and the device on which they are stored.

In [ ]:
print(f"Shape of tensor: {x_data.shape}")
print(f"Datatype of tensor: {x_data.dtype}")
print(f"Device tensor is stored on: {x_data.device}")

Other ways of writing tensors

In [ ]:
shape = (2,3)
rand_tensor = torch.rand(shape)
print(rand_tensor)

In [ ]:
ones_tensor = torch.ones(shape)
print(ones_tensor)

### Changing device of tensor

In [ ]:
if torch.cuda.is_available():
    device="cuda"

tensor = x_data.to(device)
print(tensor.device)

### Operations with tensors



#### Slicing

In [ ]:
print("Element at x_data[1,1] : ", x_data[1,1]) # element at 1, 1

In [ ]:
print("Actual value at x_data[1,1] : ", x_data[1,1].item()) # Get the actual value if only 1 element in your tensor

In [ ]:
print("First Column : ", x_data[:, 0]) # all rows, column 0

In [ ]:
print("First Row : ", x_data[1, :]) # row 1, all columns

#### Addition

In [ ]:
x = torch.ones(2, 2)
y = torch.rand(2, 2)

z=torch.add(x,y)

print(x)
print(y)
print(z)

In [ ]:
x+y

In [ ]:
z = x - y
print(z)

#### Subtraction

In [ ]:
# subtraction
z = x - y
z = torch.sub(x, y)
print(z)

#### Multiplication

In [ ]:
# multiplication
z = x * y
z = torch.mul(x,y)
print(z)

In [ ]:
x

In [ ]:
y

In [ ]:
x@y

In [ ]:
z = torch.matmul(x, y)
print(z)

#### Division

In [ ]:
# division
z = x / y
z = torch.div(x,y)
print(z)

### Defining a model

Task : Sentiment classification

    -  ("I love this movie", "POS")
    -  ("This film was terrible", "NEG")

In [ ]:
import torch.nn as nn

Defining the model class

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        fc1_output=self.fc1(x)
        fc1_output_relu=self.relu(fc1_output)
        fc2_output=self.fc2(fc1_output_relu)
        return fc2_output

` class SimpleNN(nn.Module):`  
- Defines a new neural network class called SimpleNN.
- It inherits from nn.Module, which is the base class for all PyTorch models.
- Inheriting from nn.Module gives your model features like parameter tracking, .to(device), .train(), .eval(), etc.


`super().__init__()`
- Calls the constructor of nn.Module.
- Initializes the internal machinery PyTorch needs to manage layers and parameters.

`self.fc1 = nn.Linear(input_dim, 64)`
Creates the first fully connected (dense) layer. Mathematically:
`y=Wx+b`
- Input size = input_dim
- Output size = 64

`self.relu = nn.ReLU()`
- Creates a ReLU activation function. ReLU stands for Rectified Linear Unit:

  `f(x)=max(0,x)`

### Forward Pass
Defines how data flows through the network.
- Pass input through the first linear layer.
- Apply ReLU activation.
- Pass the activated features through the second linear layer.
- Returns the final output of the network.

nn.Linear

The linear layer is a module that applies a linear transformation on the input using its stored weights and biases.

In [ ]:
x= torch.rand(128)
print(f"X: \n{x}\n\n")
print(f"Shape of x :\n{x.shape}")

fc1 = nn.Linear(in_features=128, out_features=64)
fc1_output=fc1(x)
print(f"X after linear transformation:\n{fc1_output}\n\n")
print(f"Shape of x after linear transformation :\n{fc1_output.shape}")

In [ ]:
print(f"Before ReLU: {fc1_output}\n\n")
relu = nn.ReLU()

relu_output=relu(fc1_output)
print(f"After ReLU: {relu_output}")

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        fc1_output=self.fc1(x)
        fc1_output_relu=self.relu(fc1_output)
        fc2_output=self.fc2(fc1_output_relu)
        return fc2_output

### Training the model

#### Downloading the dataset

In [ ]:
import nltk
nltk.download('movie_reviews')
from nltk.corpus import movie_reviews

In [ ]:
# for fileid in movie_reviews.fileids():
    # print(len(movie_reviews.words(fileid)))
len(movie_reviews.fileids())

In [ ]:
examples = {'pos': None, 'neg': None}

for fileid in movie_reviews.fileids():
    label = movie_reviews.categories(fileid)[0]  # 'pos' or 'neg'
    if examples[label] is None:
        text = ' '.join(movie_reviews.words(fileid)[:30])  # Show first 30 words
        examples[label] = text
    if all(examples.values()):
        break

for label, text in examples.items():
    print(f"Label: {label.upper()}")
    print(f"Review snippet: {text}...\n")


#### Transforming the data

In [ ]:
docs = []
labels = []

for fileid in movie_reviews.fileids():
    review_text = ' '.join(movie_reviews.words(fileid))
    docs.append(review_text)

    # Assign label: 1 for positive, 0 for negative
    if fileid.startswith('pos'):
        labels.append(1)
    else:
        labels.append(0)

print(docs[7])
print(labels[7])

#### Train-test splits

In [ ]:
from sklearn.model_selection import train_test_split

# 1. First split: Train (80%) and Temp (20%)
X_train_texts, X_temp_texts, y_train, y_temp = train_test_split(
    docs, labels, test_size=0.2, random_state=42)

# 2. Second split: Val (10%) and Test (10%)
X_val_texts, X_test_texts, y_val, y_test = train_test_split(
    X_temp_texts, y_temp, test_size=0.5, random_state=42)

#### Vectorisation of data

" This is a cat"
" This is a boy"
" Cat sat on a mat"





Converting into vectors using bag of words.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(max_features=2048, lowercase=True, stop_words='english')

X_train = vectorizer.fit_transform(X_train_texts).toarray()
X_test = vectorizer.transform(X_test_texts).toarray()
X_val = vectorizer.transform(X_val_texts).toarray()

#### Dataloaders

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Convert to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)


In [ ]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

#### Initialising Model, Optimiser, Loss function

In [ ]:
import torch.optim as optim

model = SimpleNN(input_dim=2048)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

#### Training loop

In [ ]:
total_epochs=5

for epoch in range(5):
    #Training
    model.train()
    total_loss, total_acc = 0, 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss = total_loss + loss.item()

    # Validation
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val_tensor)
        val_loss = criterion(val_outputs, y_val_tensor)

    print(f"Epoch {epoch+1}: Train Loss = {total_loss:.4f}, Val Loss = {val_loss:.4f}")


#### Evaluation

In [ ]:
def compute_accuracy(preds, labels):
    correct = (preds == labels)
    correct_float = correct.float()
    accuracy = correct_float.mean()
    return accuracy.item()

In [ ]:
# Final test accuracy
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    test_preds = test_outputs.argmax(1)
    test_accuracy = compute_accuracy(test_preds, y_test_tensor)

print(f"\nTest Accuracy: {test_accuracy:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Get predictions from the model on the test set
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    test_preds = test_outputs.argmax(1)

# Compute the confusion matrix
cm = confusion_matrix(y_test_tensor, test_preds)

# Visualize the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix for Sentiment Classification')
plt.show()

In [ ]:
import nltk
import torch
import torch.nn as nn
import torch.optim as optim
from nltk.corpus import movie_reviews
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
import numpy as np


nltk.download('movie_reviews')

docs = [' '.join(movie_reviews.words(fileid)) for fileid in movie_reviews.fileids()]
labels = [1 if fileid.startswith('pos') else 0 for fileid in movie_reviews.fileids()]

X_train_texts, X_test_texts, y_train, y_test = train_test_split(docs, labels, test_size=0.2, random_state=42)

# Use sklearn's CountVectorizer to tokenize and vectorize
vectorizer = CountVectorizer(max_features=2000, lowercase=True, stop_words='english')
X_train = vectorizer.fit_transform(X_train_texts).toarray()
X_test = vectorizer.transform(X_test_texts).toarray()

# Convert to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Use DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Define a simple neural network
class SimpleNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

model = SimpleNN(input_dim=2000)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
for epoch in range(5):
    model.train()
    total_loss, total_acc = 0, 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_acc += (outputs.argmax(1) == y_batch).sum().item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Accuracy: {total_acc/len(train_dataset):.4f}")

# Evaluation
model.eval()
with torch.no_grad():
    preds = model(X_test_tensor).argmax(1)
    acc = (preds == y_test_tensor).float().mean()
    print(f"\nTest Accuracy: {acc:.4f}")


In [ ]:
import random

# Choose a random index from the test set
random_index = random.randint(0, len(X_test_texts) - 1)
sample_review_text = X_test_texts[random_index]
sample_true_label = y_test[random_index]

# Vectorize the sample review text
sample_vectorized = vectorizer.transform([sample_review_text]).toarray()
sample_tensor = torch.tensor(sample_vectorized, dtype=torch.float32)

# Make a prediction with the model
model.eval() # Set the model to evaluation mode
with torch.no_grad():
    prediction_output = model(sample_tensor)
    predicted_class = prediction_output.argmax(1).item()

sentiment_map = {0: 'Negative', 1: 'Positive'}

print(f"Sample Review: {sample_review_text}")
print(f"\nTrue Sentiment: {sentiment_map[sample_true_label]}")
print(f"Predicted Sentiment: {sentiment_map[predicted_class]}")

In [ ]:
text = "I spent 3 hours in the movie theatre."

sample_vectorized = vectorizer.transform([text]).toarray()
sample_tensor = torch.tensor(sample_vectorized, dtype=torch.float32)

model.eval() # Set the model to evaluation mode
with torch.no_grad():
    prediction_output = model(sample_tensor)
    predicted_class = prediction_output.argmax(1).item()

sentiment_map = {0: 'Negative', 1: 'Positive'}

print(f"Sample Review: {text}")
# print(f"\nTrue Sentiment: {sentiment_map[sample_true_label]}")
print(f"Predicted Sentiment: {sentiment_map[predicted_class]}")